# 01 Data Loading and Time-Based Split

Этот ноутбук превращает сырые synthetic показы в аккуратно типизированный, валидированный и корректно разбитый по времени датасет для ML.

В этом ноутбуке мы:

- Загружаем `data/raw/synthetic_shows.csv`, приводим типы колонок (datetime/date/numeric/categorical/boolean) к единой схеме.
- Валидируем те же структурные ограничения, что прописаны в Pydantic‑модели `ShowRecord` (capacity, sold_tickets, occupancy_rate, release_date ≤ show_datetime и т.п.).
- Выполняем time‑based split на `train.csv`, `validation.csv`, `test.csv` по `show_datetime`, гарантируя отсутствие leakage: `max(train.show_datetime) ≤ min(validation.show_datetime) ≤ min(test.show_datetime)`.
- Сохраняем полученные сплиты в `data/processed` и печатаем краткую сводку по размеру и распределению основных признаков.


## Навигация по ноутбуку

### Зачем нужен этот ноутбук
Этот ноутбук превращает сырые показы в типизированный, валидированный и корректно разбитый по времени датасет для ML. Это переходный этап между проверкой сырого датасета и обучением модели спроса.

### Что здесь самое важное
- Зафиксировать стабильную схему данных и типы признаков.
- Построить временные train / validation / test split без утечки информации из будущего.
- Сохранить артефакты, которые затем используются всеми следующими ноутбуками.

### Какие результаты здесь ключевые
- Подготовленные датасеты в `data/processed/`.
- Воспроизводимая схема разбиения данных.
- Понимание того, какие колонки идут в признаки, а какие — в целевые переменные.

### Как читать этот ноутбук
Если меняется качество модели или поведение downstream-оптимизатора, это один из первых ноутбуков, который нужно проверить. Здесь задаётся контракт данных для всех следующих этапов.


In [1]:
from __future__ import annotations

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATASET_PATH = PROJECT_ROOT / 'data' / 'raw' / 'synthetic_shows.csv'
SCHEMA_EXAMPLE_PATH = PROJECT_ROOT / 'data' / 'schema' / 'dataset_template.csv'

PROJECT_ROOT, DATASET_PATH, SCHEMA_EXAMPLE_PATH

(PosixPath('/Users/mishatrubik/Desktop/QC/BOQ'),
 PosixPath('/Users/mishatrubik/Desktop/QC/BOQ/data/raw/synthetic_shows.csv'),
 PosixPath('/Users/mishatrubik/Desktop/QC/BOQ/data/schema/dataset_template.csv'))

In [2]:
TARGET_COLUMNS = ['sold_tickets', 'occupancy_rate']
DATETIME_COLUMNS = ['show_datetime']
DATE_COLUMNS = ['release_date']
CATEGORICAL_COLUMNS = [
    'show_id', 'cinema_id', 'hall_id', 'movie_id', 'genre', 'age_rating',
    'format', 'city', 'hour_bucket', 'runtime_bucket'
]
NUMERIC_COLUMNS = [
    'hall_capacity', 'runtime_min', 'base_price', 'sold_tickets', 'occupancy_rate',
    'day_of_week', 'month', 'release_age_days', 'lead_time_days',
    'screening_number_for_movie'
]
BOOLEAN_COLUMNS = ['is_weekend', 'holiday_flag', 'prime_time_flag']

In [3]:
def load_dataset(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    for col in DATETIME_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

    for col in DATE_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce').dt.date

    for col in NUMERIC_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    for col in BOOLEAN_COLUMNS:
        if col in df.columns:
            df[col] = df[col].astype('boolean')

    for col in CATEGORICAL_COLUMNS:
        if col in df.columns:
            df[col] = df[col].astype('string').str.strip()

    return df

In [4]:
def validate_dataset(df: pd.DataFrame) -> tuple[list[str], pd.DataFrame]:
    errors: list[str] = []

    required_columns = [
        'show_id', 'cinema_id', 'hall_id', 'hall_capacity', 'movie_id', 'genre',
        'age_rating', 'runtime_min', 'release_date', 'show_datetime', 'format',
        'base_price', 'sold_tickets', 'occupancy_rate'
    ]

    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        errors.append(f'Missing columns: {missing}')

    if 'hall_capacity' in df.columns and (df['hall_capacity'] <= 0).any():
        errors.append('hall_capacity must be > 0 for all rows')

    if 'sold_tickets' in df.columns and (df['sold_tickets'] < 0).any():
        errors.append('sold_tickets must be >= 0 for all rows')

    if {'sold_tickets', 'hall_capacity'}.issubset(df.columns):
        bad = df['sold_tickets'] > df['hall_capacity']
        if bad.any():
            errors.append(f'sold_tickets exceeds hall_capacity in {int(bad.sum())} rows')

    if {'sold_tickets', 'hall_capacity', 'occupancy_rate'}.issubset(df.columns):
        expected = df['sold_tickets'] / df['hall_capacity']
        mismatch = (expected - df['occupancy_rate']).abs() > 1e-3
        if mismatch.any():
            errors.append(f'occupancy_rate mismatch in {int(mismatch.sum())} rows')

    if {'release_date', 'show_datetime'}.issubset(df.columns):
        release_ts = pd.to_datetime(df['release_date'], errors='coerce')
        invalid_dates = release_ts > df['show_datetime']
        if invalid_dates.any():
            errors.append(f'release_date later than show_datetime in {int(invalid_dates.sum())} rows')

    quality_report = pd.DataFrame({
        'column': df.columns,
        'null_count': [int(df[c].isna().sum()) for c in df.columns],
        'dtype': [str(df[c].dtype) for c in df.columns],
    }).sort_values(['null_count', 'column'], ascending=[False, True])

    return errors, quality_report

In [5]:
df = load_dataset(DATASET_PATH)
errors, quality_report = validate_dataset(df)

print('dataset source:', DATASET_PATH)
print('schema example:', SCHEMA_EXAMPLE_PATH)
print('rows:', len(df))
print('columns:', len(df.columns))
print('errors:', errors if errors else 'none')

df.head()

dataset source: /Users/mishatrubik/Desktop/QC/BOQ/data/raw/synthetic_shows.csv
schema example: /Users/mishatrubik/Desktop/QC/BOQ/data/schema/dataset_template.csv
rows: 898
columns: 25
errors: ['release_date later than show_datetime in 12 rows']


,show_id,cinema_id,hall_id,hall_capacity,movie_id,genre,age_rating,runtime_min,release_date,show_datetime,...,day_of_week,is_weekend,hour_bucket,month,holiday_flag,release_age_days,lead_time_days,prime_time_flag,runtime_bucket,screening_number_for_movie
0,show_00001,cinema_msk_01,hall_1,120,movie_family_01,animation,6+,92,2026-02-20,2026-03-01 10:30:00,...,6,True,morning,3,False,9,5.5,False,medium,1
1,show_00002,cinema_msk_01,hall_1,120,movie_family_01,animation,6+,92,2026-02-20,2026-03-01 14:15:00,...,6,True,day,3,False,9,13.8,True,medium,2
2,show_00003,cinema_msk_01,hall_2,80,movie_comedy_01,comedy,12+,105,2026-02-10,2026-03-01 10:45:00,...,6,True,morning,3,False,19,16.7,False,medium,1
3,show_00004,cinema_msk_01,hall_2,80,movie_dune_2,sci_fi,16+,166,2026-03-01,2026-03-01 14:30:00,...,6,True,day,3,False,0,9.8,True,long,1
4,show_00005,cinema_msk_01,hall_2,80,movie_dune_2,sci_fi,16+,166,2026-03-01,2026-03-01 18:45:00,...,6,True,evening,3,False,0,4.4,True,long,2


In [6]:
def time_based_split(
    df: pd.DataFrame,
    datetime_col: str = 'show_datetime',
    train_ratio: float = 0.7,
    val_ratio: float = 0.15,
):
    ordered = df.sort_values(datetime_col).reset_index(drop=True).copy()
    n = len(ordered)

    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_df = ordered.iloc[:train_end].copy()
    val_df = ordered.iloc[train_end:val_end].copy()
    test_df = ordered.iloc[val_end:].copy()

    return train_df, val_df, test_df

In [7]:
train_df, val_df, test_df = time_based_split(df)

split_summary = pd.DataFrame([
    {
        'split': 'train',
        'rows': len(train_df),
        'min_datetime': train_df['show_datetime'].min() if len(train_df) else None,
        'max_datetime': train_df['show_datetime'].max() if len(train_df) else None,
    },
    {
        'split': 'validation',
        'rows': len(val_df),
        'min_datetime': val_df['show_datetime'].min() if len(val_df) else None,
        'max_datetime': val_df['show_datetime'].max() if len(val_df) else None,
    },
    {
        'split': 'test',
        'rows': len(test_df),
        'min_datetime': test_df['show_datetime'].min() if len(test_df) else None,
        'max_datetime': test_df['show_datetime'].max() if len(test_df) else None,
    },
])

split_summary

,split,rows,min_datetime,max_datetime
0,train,628,2026-03-01 10:15:00,2026-04-12 11:45:00
1,validation,135,2026-04-12 14:00:00,2026-04-22 11:00:00
2,test,135,2026-04-22 11:15:00,2026-04-30 21:15:00


In [8]:
leakage_checks = {
    'train_max_leq_val_min': bool(train_df['show_datetime'].max() <= val_df['show_datetime'].min()),
    'val_max_leq_test_min': bool(val_df['show_datetime'].max() <= test_df['show_datetime'].min()),
    'train_max_leq_test_min': bool(train_df['show_datetime'].max() <= test_df['show_datetime'].min()),
}

leakage_checks

{'train_max_leq_val_min': True,
 'val_max_leq_test_min': True,
 'train_max_leq_test_min': True}

In [9]:
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(OUTPUT_DIR / 'train.csv', index=False)
val_df.to_csv(OUTPUT_DIR / 'validation.csv', index=False)
test_df.to_csv(OUTPUT_DIR / 'test.csv', index=False)

print('saved to', OUTPUT_DIR)

saved to /Users/mishatrubik/Desktop/QC/BOQ/data/processed
